In [ ]:
# ASSIGNMENT 3 – CO3
from pathlib import Path
import subprocess
import zipfile
import shutil

PROJECT = Path("/content/CSA09-192424008-Assignment3")
PROJECT.mkdir(parents=True, exist_ok=True)

if shutil.which("javac") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "openjdk-17-jdk-headless", "-qq"],
        check=True
    )


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task01_ATM/ATMSimulation.java
import java.util.HashMap;
import java.util.InputMismatchException;
import java.util.Map;
import java.util.Scanner;

class InsufficientFundsException extends Exception {
    InsufficientFundsException(String message) { super(message); }
}
class AccountNotFoundException extends Exception {
    AccountNotFoundException(String message) { super(message); }
}

public class ATMSimulation {
    private static final Map<Integer, Double> accounts = new HashMap<>();
    static {
        accounts.put(1001, 5000.0);
        accounts.put(1002, 1200.0);
    }

    static void withdraw(int account, double amount)
            throws AccountNotFoundException, InsufficientFundsException {
        if (amount <= 0) throw new IllegalArgumentException("Amount must be positive");
        Double balance = accounts.get(account);
        if (balance == null) throw new AccountNotFoundException("Account not found");
        if (amount > balance)
            throw new InsufficientFundsException("Insufficient account balance");
        accounts.put(account, balance - amount);
        System.out.println("Please collect cash: Rs." + amount);
        System.out.println("Remaining balance: Rs." + accounts.get(account));
    }

    public static void main(String[] args) {
        System.out.println("ATM session started");
        try (Scanner sc = new Scanner(System.in)) {
            System.out.print("Enter account number (1001 or 1002): ");
            int account = sc.nextInt();
            System.out.print("Enter withdrawal amount: ");
            double amount = sc.nextDouble();

            try {
                withdraw(account, amount);
            } catch (AccountNotFoundException | InsufficientFundsException e) {
                System.out.println("Transaction declined: " + e.getMessage());
            }
        } catch (InputMismatchException e) {
            System.out.println("Invalid input: enter numeric values only.");
        } catch (IllegalArgumentException e) {
            System.out.println("Invalid withdrawal: " + e.getMessage());
        } catch (Exception e) {
            System.out.println("ATM service error: " + e.getMessage());
        } finally {
            System.out.println("ATM session closed. Please take your card.");
        }
    }
}


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task02_Exceptions/StorageDemo.java
import java.io.IOException;

class ExamException extends Exception {
    ExamException(String message) { super(message); }
    ExamException(String message, Throwable cause) { super(message, cause); }
}
class ExamWindowClosedException extends ExamException {
    ExamWindowClosedException(String message) { super(message); }
}
class StudentNotRegisteredException extends ExamException {
    StudentNotRegisteredException(String message) { super(message); }
}
class SubmissionStorageException extends ExamException {
    SubmissionStorageException(String message, Throwable cause) {
        super(message, cause);
    }
}
class InvalidSubmissionRequestException extends RuntimeException {
    InvalidSubmissionRequestException(String message) { super(message); }
}

class Storage {
    void save(String fileName, long sizeKb) throws SubmissionStorageException {
        try {
            if (sizeKb > 500) throw new IOException("Disk quota exceeded");
            System.out.println("Saved " + fileName);
        } catch (IOException e) {
            throw new SubmissionStorageException("Could not store " + fileName, e);
        }
    }
}
class ExamService {
    private final Storage storage = new Storage();

    void submit(String studentId, boolean registered, boolean examOpen,
                String fileName, long sizeKb) throws ExamException {
        if (studentId == null || studentId.trim().isEmpty())
            throw new InvalidSubmissionRequestException("Student ID is required");
        if (!examOpen) throw new ExamWindowClosedException("Exam window is closed");
        if (!registered)
            throw new StudentNotRegisteredException("Student is not registered");
        storage.save(fileName, sizeKb);
    }
}
public class StorageDemo {
    public static void main(String[] args) {
        ExamService service = new ExamService();
        try {
            service.submit("STU001", true, true, "answer01.pdf", 120);
            service.submit("STU002", true, true, "answer02.pdf", 800);
        } catch (ExamException e) {
            System.out.println("Portal message: " + e.getMessage());
            if (e.getCause() != null)
                System.out.println("Original cause: " + e.getCause().getMessage());
            e.printStackTrace();
        } catch (InvalidSubmissionRequestException e) {
            System.out.println("Invalid request: " + e.getMessage());
        }
    }
}


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task03_Resources/ManualSalesReport.java
import java.io.*;

public class ManualSalesReport {
    public static void generate(String[] files, String output) {
        BufferedWriter writer = null;
        try {
            writer = new BufferedWriter(new FileWriter(output));
            long total = 0;
            for (String file : files) {
                BufferedReader reader = null;
                try {
                    reader = new BufferedReader(new FileReader(file));
                    String line;
                    while ((line = reader.readLine()) != null)
                        total += Long.parseLong(line.trim());
                } catch (IOException | NumberFormatException e) {
                    System.err.println("Failed file " + file + ": " + e.getMessage());
                    throw new RuntimeException("Report incomplete", e);
                } finally {
                    if (reader != null) {
                        try { reader.close(); }
                        catch (IOException closeError) {
                            System.err.println("Close failed: " + closeError.getMessage());
                        }
                    }
                }
            }
            writer.write("TOTAL=" + total);
        } catch (IOException | RuntimeException e) {
            System.err.println("Report generation failed: " + e.getMessage());
        } finally {
            if (writer != null) {
                try { writer.close(); }
                catch (IOException e) {
                    System.err.println("Output close failed: " + e.getMessage());
                }
            }
        }
    }

    public static void main(String[] args) {
        String[] files = args.length > 0 ? args :
            new String[]{"test-files/store1.txt", "test-files/store2.txt"};
        generate(files, "manual-summary.txt");
        System.out.println("Manual report attempt finished.");
    }
}


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task03_Resources/SafeSalesReport.java
import java.io.*;

public class SafeSalesReport {
    public static void generate(String[] files, String output) {
        long total = 0;
        try (BufferedWriter writer = new BufferedWriter(new FileWriter(output))) {
            for (String file : files) {
                System.out.println("Processing: " + file);
                try (BufferedReader reader = new BufferedReader(new FileReader(file))) {
                    String line;
                    while ((line = reader.readLine()) != null)
                        total += Long.parseLong(line.trim());
                } catch (IOException | NumberFormatException e) {
                    throw new RuntimeException("Could not process " + file, e);
                }
            }
            writer.write("TOTAL=" + total);
        } catch (IOException | RuntimeException e) {
            System.err.println("Report failed: " + e.getMessage());
        }
    }

    public static void main(String[] args) {
        String[] files = args.length > 0 ? args :
            new String[]{"test-files/store1.txt", "test-files/store2.txt"};
        generate(files, "safe-summary.txt");
        System.out.println("Safe report attempt finished.");
    }
}


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task04_Threads/RunnableStations.java
class ScannerTask implements Runnable {
    private final int station, scripts;
    ScannerTask(int station, int scripts) {
        this.station = station; this.scripts = scripts;
    }
    public void run() {
        try {
            for (int i = 0; i < scripts; i++) Thread.sleep(10);
            System.out.println("Station " + station + " scanned " + scripts);
        } catch (InterruptedException e) {
            Thread.currentThread().interrupt();
        }
    }
}
public class RunnableStations {
    public static void main(String[] args) throws InterruptedException {
        Thread[] stations = new Thread[4];
        for (int i = 0; i < 4; i++) {
            stations[i] = new Thread(new ScannerTask(i + 1, 5));
            stations[i].start();
        }
        for (Thread t : stations) t.join();
        System.out.println("All stations completed.");
    }
}


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task04_Threads/ThreadExperiments.java
public class ThreadExperiments {
    public static void main(String[] args) throws InterruptedException {
        Thread high = new Thread(() -> work("HIGH"));
        Thread low = new Thread(() -> work("LOW"));
        high.setPriority(Thread.MAX_PRIORITY);
        low.setPriority(Thread.MIN_PRIORITY);

        high.start();
        low.start();
        System.out.println("High thread alive? " + high.isAlive());
        System.out.println("Low thread alive? " + low.isAlive());
        high.join();
        low.join();

        Thread daemon = new Thread(() -> {
            try {
                Thread.sleep(1000);
                System.out.println("Daemon task finished");
            } catch (InterruptedException e) {
                Thread.currentThread().interrupt();
            }
        });
        daemon.setDaemon(true); // Must be set before start().
        daemon.start();
        System.out.println("Main completed; daemon may be stopped when JVM exits.");
    }

    private static void work(String label) {
        for (int i = 1; i <= 5; i++)
            System.out.println(label + " thread step " + i);
    }
}


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task04_Threads/ThreadStations.java
class ScannerStationThread extends Thread {
    private final int station, scripts;
    ScannerStationThread(int station, int scripts) {
        this.station = station; this.scripts = scripts;
    }
    public void run() {
        try {
            for (int i = 0; i < scripts; i++) Thread.sleep(10);
            System.out.println("Station " + station + " scanned " + scripts);
        } catch (InterruptedException e) {
            Thread.currentThread().interrupt();
        }
    }
}
public class ThreadStations {
    public static void main(String[] args) throws InterruptedException {
        Thread[] stations = new Thread[4];
        for (int i = 0; i < 4; i++) {
            stations[i] = new ScannerStationThread(i + 1, 5);
            stations[i].start();
        }
        for (Thread t : stations) t.join();
        System.out.println("All stations completed.");
    }
}


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task05_Synchronization/BookingRaceExperiment.java
public class BookingRaceExperiment {
    public static void main(String[] args) throws InterruptedException {
        for (int trial = 1; trial <= 10; trial++) {
            SeatBooking screen = new SeatBooking(10);
            Thread[] users = new Thread[100];
            long start = System.nanoTime();

            for (int i = 0; i < users.length; i++) {
                final int seat = i % 10;
                users[i] = new Thread(() -> screen.bookUnsafe(seat));
                users[i].start();
            }
            for (Thread t : users) t.join();

            long elapsed = System.nanoTime() - start;
            System.out.printf("Trial %d: unsafe seats sold=%d, elapsed=%.3f ms%n",
                    trial, screen.getBookingsMade(), elapsed / 1_000_000.0);
        }
    }
}


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task05_Synchronization/SeatBooking.java
public class SeatBooking {
    private final boolean[] booked;
    private int bookingsMade = 0;

    SeatBooking(int seats) { booked = new boolean[seats]; }

    // Intentionally unsafe version to demonstrate a race condition.
    public boolean bookUnsafe(int seat) {
        if (!booked[seat]) {
            Thread.yield();
            booked[seat] = true;
            bookingsMade++;
            return true;
        }
        return false;
    }

    public synchronized boolean book(int seat) {
        if (booked[seat]) return false;
        booked[seat] = true;
        bookingsMade++;
        return true;
    }

    public boolean bookWithBlock(int seat) {
        synchronized (this) {
            if (booked[seat]) return false;
            booked[seat] = true;
            bookingsMade++;
            return true;
        }
    }

    public int getBookingsMade() { return bookingsMade; }

    public static void main(String[] args) throws InterruptedException {
        runTrial("Synchronized method", 1);
        runTrial("Synchronized block", 2);
    }

    private static void runTrial(String label, int mode) throws InterruptedException {
        SeatBooking screen = new SeatBooking(10);
        Thread[] users = new Thread[100];
        for (int i = 0; i < users.length; i++) {
            final int seat = i % 10;
            users[i] = new Thread(() -> {
                if (mode == 1) screen.book(seat);
                else screen.bookWithBlock(seat);
            });
            users[i].start();
        }
        for (Thread t : users) t.join();
        System.out.println(label + " – seats sold: " + screen.getBookingsMade());
    }
}


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task06_Deadlock_ProducerConsumer/TokenCounter.java
public class TokenCounter {
    private final int[] buffer = new int[5];
    private int count = 0, in = 0, out = 0;

    public synchronized void put(int token) throws InterruptedException {
        while (count == buffer.length) wait();
        buffer[in] = token;
        in = (in + 1) % buffer.length;
        count++;
        notifyAll();
    }

    public synchronized int take() throws InterruptedException {
        while (count == 0) wait();
        int token = buffer[out];
        out = (out + 1) % buffer.length;
        count--;
        notifyAll();
        return token;
    }

    public static void main(String[] args) throws InterruptedException {
        TokenCounter counter = new TokenCounter();
        int[] collected = new int[2];

        Thread cook = new Thread(() -> {
            try {
                for (int token = 1; token <= 20; token++) counter.put(token);
            } catch (InterruptedException e) {
                Thread.currentThread().interrupt();
            }
        });

        Thread[] students = new Thread[2];
        for (int s = 0; s < 2; s++) {
            final int id = s;
            students[s] = new Thread(() -> {
                try {
                    for (int k = 0; k < 10; k++) collected[id] += counter.take();
                } catch (InterruptedException e) {
                    Thread.currentThread().interrupt();
                }
            });
        }

        cook.start();
        for (Thread student : students) student.start();
        cook.join();
        for (Thread student : students) student.join();

        System.out.println("Sum of tokens collected: "
                + (collected[0] + collected[1]) + " (expected 210)");
    }
}


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task06_Deadlock_ProducerConsumer/Transfers.java
class Wallet {
    final int id;
    int balance;
    Wallet(int id, int balance) { this.id = id; this.balance = balance; }
}
public class Transfers {
    // Safe strategy: every transfer locks wallets in ascending ID order.
    static void transferSafe(Wallet from, Wallet to, int amount) {
        if (amount <= 0) throw new IllegalArgumentException("Amount must be positive");
        Wallet first = from.id < to.id ? from : to;
        Wallet second = from.id < to.id ? to : from;
        synchronized (first) {
            synchronized (second) {
                if (from.balance < amount)
                    throw new IllegalArgumentException("Insufficient balance");
                from.balance -= amount;
                to.balance += amount;
            }
        }
    }

    public static void main(String[] args) {
        Wallet a = new Wallet(1, 1000);
        Wallet b = new Wallet(2, 500);
        transferSafe(a, b, 200);
        System.out.println("Wallet A: " + a.balance);
        System.out.println("Wallet B: " + b.balance);
        System.out.println("Total: " + (a.balance + b.balance));
    }
}


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task07_ConcurrentProcessor/ConcurrentResultProcessor.java
import java.io.BufferedWriter;
import java.io.IOException;
import java.nio.file.Files;
import java.nio.file.Path;
import java.nio.file.Paths;
import java.util.ArrayList;
import java.util.List;
import java.util.concurrent.Callable;
import java.util.concurrent.ConcurrentLinkedQueue;
import java.util.concurrent.ExecutionException;
import java.util.concurrent.ExecutorService;
import java.util.concurrent.Executors;
import java.util.concurrent.Future;
import java.util.concurrent.TimeUnit;

class FileResult {
    final String file, studentId, status, message;
    final int total;

    FileResult(String file, String studentId, String status, int total, String message) {
        this.file = file;
        this.studentId = studentId;
        this.status = status;
        this.total = total;
        this.message = message;
    }

    public String toString() {
        return file + " | " + status + " | " + studentId + " | " + total + " | " + message;
    }
}

class ScoreTask implements Callable<FileResult> {
    private final Path path;

    ScoreTask(Path path) { this.path = path; }

    public FileResult call() {
        try {
            List<String> lines = Files.readAllLines(path);
            if (lines.size() < 2)
                throw new IllegalArgumentException("Missing score data");

            String studentId = lines.get(0).trim();
            if (studentId.isEmpty())
                throw new IllegalArgumentException("Missing student ID");

            int total = 0;
            for (int i = 1; i < lines.size(); i++) {
                int mark = Integer.parseInt(lines.get(i).trim());
                if (mark < 0 || mark > 100)
                    throw new IllegalArgumentException("Score outside 0..100");
                total += mark;
            }
            return new FileResult(path.getFileName().toString(),
                    studentId, "SUCCESS", total, "validated");
        } catch (IOException | RuntimeException e) {
            return new FileResult(path.getFileName().toString(),
                    "-", "FAILURE", 0, e.getMessage());
        }
    }
}

public class ConcurrentResultProcessor {
    static void run(Path folder, int poolSize) throws Exception {
        Path manifest = folder.resolve("manifest.txt");
        List<String> names = Files.readAllLines(manifest);
        List<Path> expectedFiles = new ArrayList<>();
        for (String name : names) {
            String trimmed = name.trim();
            if (!trimmed.isEmpty()) expectedFiles.add(folder.resolve(trimmed));
        }

        ExecutorService pool = Executors.newFixedThreadPool(poolSize);
        ConcurrentLinkedQueue<FileResult> results = new ConcurrentLinkedQueue<>();
        List<Future<FileResult>> futures = new ArrayList<>();
        long start = System.nanoTime();

        try {
            for (Path file : expectedFiles)
                futures.add(pool.submit(new ScoreTask(file)));

            for (Future<FileResult> future : futures) {
                try {
                    results.add(future.get());
                } catch (ExecutionException e) {
                    results.add(new FileResult("unknown", "-", "FAILURE",
                            0, String.valueOf(e.getCause())));
                }
            }
        } finally {
            pool.shutdown();
            if (!pool.awaitTermination(1, TimeUnit.MINUTES))
                pool.shutdownNow();
        }

        long elapsed = System.nanoTime() - start;
        long success = results.stream().filter(r -> r.status.equals("SUCCESS")).count();
        long failure = results.size() - success;

        System.out.println("Files expected: " + expectedFiles.size());
        System.out.println("Success: " + success);
        System.out.println("Failure: " + failure);
        System.out.println("Total outcomes: " + results.size());
        System.out.printf("Elapsed: %.3f ms (pool=%d)%n", elapsed / 1e6, poolSize);

        if (results.size() != expectedFiles.size())
            throw new IllegalStateException("Incomplete summary");

        Path summary = folder.resolve("summary.txt");
        try (BufferedWriter writer = Files.newBufferedWriter(summary)) {
            for (FileResult result : results)
                writer.write(result + System.lineSeparator());
        }
    }

    public static void main(String[] args) throws Exception {
        Path folder = Paths.get(args.length > 0 ? args[0] : "scores");
        int poolSize = args.length > 1 ? Integer.parseInt(args[1]) : 4;
        if (poolSize < 1) throw new IllegalArgumentException("Pool size must be >= 1");
        run(folder, poolSize);
    }
}


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task03_Resources/test-files/store1.txt
1200
800
450


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task03_Resources/test-files/store2.txt
950
600


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task07_ConcurrentProcessor/manifest.txt
S001.txt
S002.txt
S003.txt
S004.txt
S005_missing.txt


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task07_ConcurrentProcessor/scores/S001.txt
STU001
80
75
90


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task07_ConcurrentProcessor/scores/S002.txt
STU002
60
70


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task07_ConcurrentProcessor/scores/S003.txt
STU003
80
BAD


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task07_ConcurrentProcessor/scores/S004.txt
STU004
120


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/README.md
# Java Assignment 3 – CO3

Repository: `CSA09-192424008-Assignment3`

This repository contains Java source files and sample input files for all seven Assignment 3 tasks.

## Requirements
- JDK 17 or later recommended.
- Open a terminal in the relevant task folder.
- Compile the source file with `javac`.
- Run the class with `java`.

## Folder structure

- `Task01_ATM` – ATM exception handling and custom checked exceptions.
- `Task02_Exceptions` – user-defined exceptions, exception propagation and cause chaining.
- `Task03_Resources` – manual cleanup and try-with-resources report generation.
- `Task04_Threads` – Thread, Runnable, priorities, daemon and lifecycle experiments.
- `Task05_Synchronization` – unsafe race condition and synchronized booking.
- `Task06_Deadlock_ProducerConsumer` – safe wallet transfers and bounded producer-consumer buffer.
- `Task07_ConcurrentProcessor` – ExecutorService, Callable, Future and manifest-based processing.

## Compile and run

Run commands from inside each task folder.

### Task 1 – ATM
```bash
javac ATMSimulation.java
java ATMSimulation
```
Sample valid account numbers: `1001` (balance Rs. 5000) and `1002` (balance Rs. 1200).

### Task 2 – Exception propagation
```bash
javac StorageDemo.java
java StorageDemo
```
The second sample submission is intentionally larger than 500 KB to demonstrate exception chaining.

### Task 3 – Resource handling
```bash
javac ManualSalesReport.java SafeSalesReport.java
java ManualSalesReport
java SafeSalesReport
```
Both programs read `test-files/store1.txt` and `test-files/store2.txt` relative to the current task folder.

### Task 4 – Threads
```bash
javac ThreadStations.java RunnableStations.java ThreadExperiments.java
java ThreadStations
java RunnableStations
java ThreadExperiments
```
Thread completion order can differ between runs. Priority is only a scheduling hint.

### Task 5 – Seat booking
```bash
javac SeatBooking.java BookingRaceExperiment.java
java SeatBooking
java BookingRaceExperiment
```
The unsafe experiment is timing-dependent. Synchronized booking should never sell more than ten unique seats.

### Task 6 – Deadlock and producer-consumer
```bash
javac Transfers.java TokenCounter.java
java Transfers
java TokenCounter
```
The wallet example uses a consistent lock order. The token counter uses a bounded buffer of capacity five.

### Task 7 – Concurrent result processor
Run from inside `Task07_ConcurrentProcessor`:
```bash
javac ConcurrentResultProcessor.java
java ConcurrentResultProcessor . 4
```
The program reads expected filenames from `manifest.txt`. A missing or malformed file is recorded as a failure without stopping the remaining tasks.

Run pool-size experiments:
```bash
java ConcurrentResultProcessor . 1
java ConcurrentResultProcessor . 2
java ConcurrentResultProcessor . 4
java ConcurrentResultProcessor . 8
```
Record actual elapsed times in `measurements/`. Do not invent timing values.

## Important notes
- `Task07_ConcurrentProcessor/scores/S005_missing.txt` is intentionally absent. It appears in the manifest so the processor can demonstrate missing-file handling.
- Generated summary files and console output screenshots are not pre-filled. Run the programs and add your own outputs to the `outputs/` or `measurements/` folders.
- The programs are educational demonstrations and are not production banking, examination, cinema, or clinical systems.


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task01_ATM/outputs/README.txt
Add your own console output screenshots or captured results here.


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task02_Exceptions/outputs/README.txt
Add your own console output screenshots or captured results here.


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task04_Threads/outputs/README.txt
Add your own console output screenshots or captured results here.


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task05_Synchronization/outputs/README.txt
Add your own console output screenshots or captured results here.


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task06_Deadlock_ProducerConsumer/outputs/README.txt
Add your own console output screenshots or captured results here.


In [ ]:
%%writefile /content/CSA09-192424008-Assignment3/Task07_ConcurrentProcessor/measurements/README.txt
Record measured run times for pool sizes 1, 2, 4 and 8 here. Use identical input data for each run.


In [ ]:
TASK_FOLDERS = [
    "Task01_ATM",
    "Task02_Exceptions",
    "Task03_Resources",
    "Task04_Threads",
    "Task05_Synchronization",
    "Task06_Deadlock_ProducerConsumer",
    "Task07_ConcurrentProcessor"
]

for folder in TASK_FOLDERS:
    java_sources = sorted(str(p) for p in (PROJECT / folder).glob("*.java"))
    result = subprocess.run(
        ["javac"] + java_sources,
        cwd=PROJECT,
        capture_output=True,
        text=True
    )
    print(folder, "COMPILED" if result.returncode == 0 else "COMPILATION ERROR")
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode:
        raise RuntimeError(f"Compilation failed in {folder}")


In [ ]:
result = subprocess.run(
    ["java", "-cp", str(PROJECT / "Task01_ATM"), "ATMSimulation"],
    input="1001\n500\n",
    capture_output=True,
    text=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)


In [ ]:
result = subprocess.run(
    ["java", "-cp", str(PROJECT / "Task02_Exceptions"), "StorageDemo"],
    cwd=PROJECT,
    capture_output=True,
    text=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)


In [ ]:
task_dir = PROJECT / "Task03_Resources"
for class_name in ["ManualSalesReport", "SafeSalesReport"]:
    result = subprocess.run(
        ["java", class_name],
        cwd=task_dir,
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)


In [ ]:
for class_name in ["ThreadStations", "RunnableStations", "ThreadExperiments"]:
    result = subprocess.run(
        ["java", "-cp", str(PROJECT / "Task04_Threads"), class_name],
        cwd=PROJECT,
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)


In [ ]:
for class_name in ["SeatBooking", "BookingRaceExperiment"]:
    result = subprocess.run(
        ["java", "-cp", str(PROJECT / "Task05_Synchronization"), class_name],
        cwd=PROJECT,
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)


In [ ]:
for class_name in ["Transfers", "TokenCounter"]:
    result = subprocess.run(
        ["java", "-cp", str(PROJECT / "Task06_Deadlock_ProducerConsumer"), class_name],
        cwd=PROJECT,
        capture_output=True,
        text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)


In [ ]:
task_dir = PROJECT / "Task07_ConcurrentProcessor"
result = subprocess.run(
    ["java", "ConcurrentResultProcessor", ".", "4"],
    cwd=task_dir,
    capture_output=True,
    text=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode:
    raise RuntimeError("Concurrent result processor failed")


In [ ]:
task_dir = PROJECT / "Task07_ConcurrentProcessor"
for pool_size in [1, 2, 4, 8]:
    result = subprocess.run(
        ["java", "ConcurrentResultProcessor", ".", str(pool_size)],
        cwd=task_dir,
        capture_output=True,
        text=True
    )
    print(f"POOL SIZE {pool_size}")
    print(result.stdout)
    if result.stderr:
        print(result.stderr)


In [ ]:
ZIP_PATH = Path("/content/CSA09-192424008-Assignment3.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in PROJECT.rglob("*"):
        if path.is_file() and path.suffix != ".class":
            archive.write(path, path.relative_to(PROJECT.parent))
print(ZIP_PATH)
